# 第 20 章 · 收尾：把学习策略部署成 ROS 2 节点

本章回答：一个 VLA / 模仿学习策略，怎么包成 ROS 2 节点接到真机上？图像和关节状态怎么对齐时间？几百毫秒一次的推理，怎么变成 50 Hz 以上平滑的关节命令？怎么保证出问题时机器人安全停下？延迟怎么量？

配套讲义：仓库里 `ros2/notes/lessons/20-policy-node.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 Python 标准库，不需要 ROS 2。

**为什么是模拟**：本机没有 ROS 2 也没有真模型。时间戳、动作块、延迟样本都是编的，只用来把讲义第 1–3 步的规则跑一遍。

In [ ]:
import math

print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 规则 / 数字 | 写法 | 跑出来 |
|---|---|---|---|
| 第 1 步 近似时间同步 | 按时间戳配对，差太多不配 | `approx_sync(IMAGES, JOINTS, 0.005)` | 前三张配上，0.100 那张没配上 |
| 第 2 步 动作块缓冲区 | 按当前时刻取第几步 | `buf.step_at(now)` | 1.000→0；1.019→0；1.020→1；1.050→2 |
| 第 2 步 时间集成 | 同一时刻多个块取平均 | `temporal_ensemble([0.16, 0.20])` | 0.18 |
| 第 3 步 限幅 | 每步变化不超过限值 | `limit_step(0.16, 0.40, 0.05)` | 0.21 |
| 第 3 步 测量 | 看平均，更要看 p99 | `percentile(latency_ms, 99)` | 平均 51.3、p50 44、p99 120 ms |

## 1. 先把要用的东西定义好

这几格只是定义（常量、函数、类），运行后没有输出也正常。用到的前面章节的函数也一起抄进来了，不需要本地文件。

**常量 `STEP_S`**

In [ ]:
STEP_S = 0.02  # 动作块每步 20 ms（讲义第 2 步：未来 50 步，每步 20 ms）

**类 `ChunkBuffer`**：推理写、执行读：存"最新动作块 + 它对应的起始时刻"，执行时按当前时刻算出该取第几步。

In [ ]:
class ChunkBuffer:
    """推理写、执行读：存"最新动作块 + 它对应的起始时刻"，执行时按当前时刻算出该取第几步。"""

    def __init__(self):
        self.chunk: list[float] = []
        self.start = 0.0

    def write(self, chunk: list[float], start: float):
        self.chunk, self.start = chunk, start

    def step_at(self, now: float) -> int:
        return math.floor((now - self.start) / STEP_S + 1e-9)

    def action_at(self, now: float) -> float:
        return self.chunk[min(self.step_at(now), len(self.chunk) - 1)]

**常量 `IMAGES`**

In [ ]:
IMAGES = [0.000, 0.033, 0.066, 0.100]  # 图像时间戳，30 Hz（编的）

**常量 `JOINTS`**

In [ ]:
JOINTS = [0.002 + 0.010 * k for k in range(10)]  # 关节状态时间戳，100 Hz（编的）

**函数 `approx_sync`**：近似时间同步：每张图配一条时间戳最近的关节状态，相差超过 slop 秒就不配。

In [ ]:
def approx_sync(images: list[float], joints: list[float], slop: float) -> list[tuple[float, float]]:
    """近似时间同步：每张图配一条时间戳最近的关节状态，相差超过 slop 秒就不配。"""
    pairs = []
    for ti in images:
        tj = min(joints, key=lambda t: abs(t - ti))
        if abs(tj - ti) <= slop:
            pairs.append((ti, tj))
    return pairs

**函数 `limit_step`**：限幅：每步关节变化量不超过 max_delta。

In [ ]:
def limit_step(prev: float, target: float, max_delta: float) -> float:
    """限幅：每步关节变化量不超过 max_delta。"""
    return prev + max(-max_delta, min(max_delta, target - prev))

**函数 `percentile`**：第 q 百分位（最近秩法）：从小到大排，取第 ⌈q/100 · n⌉ 个。

In [ ]:
def percentile(samples: list[float], q: float) -> float:
    """第 q 百分位（最近秩法）：从小到大排，取第 ⌈q/100 · n⌉ 个。"""
    s = sorted(samples)
    return s[math.ceil(q / 100 * len(s)) - 1]

**函数 `temporal_ensemble`**：时间集成：同一时刻几个动作块给出的动作取平均（ACT 用指数权重，这里取等权，简化）。

In [ ]:
def temporal_ensemble(predictions: list[float]) -> float:
    """时间集成：同一时刻几个动作块给出的动作取平均（ACT 用指数权重，这里取等权，简化）。"""
    return sum(predictions) / len(predictions)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 · 近似时间同步：图像 30 Hz、关节 100 Hz，差 5 ms 以内才配对")
for ti, tj in approx_sync(IMAGES, JOINTS, slop=0.005):
    print(f"  图像 {ti:.3f} s ↔ 关节 {tj:.3f} s（差 {abs(ti - tj) * 1000:.0f} ms）")
print("  0.100 s 那张图没配上：最近的关节状态差了 8 ms")

In [ ]:
print("\n第二步 · 动作块缓冲区：块从 t=1.00 s 开始，每步 20 ms；执行回调按当前时刻取第几步")
buf = ChunkBuffer()
buf.write([0.10, 0.12, 0.14, 0.16, 0.18], start=1.00)  # 5 步的动作块（编的）
for now in [1.00, 1.019, 1.02, 1.05]:
    print(f"  now={now:.3f} s → 第 {buf.step_at(now)} 步，动作 {buf.action_at(now):.2f}")

In [ ]:
print("\n第二步（续）· 衔接：同一时刻旧块说 0.16、新块说 0.20，时间集成取平均")
print("  硬切换：0.16 → 0.20 一下跳 0.04；时间集成:", round(temporal_ensemble([0.16, 0.20]), 3))

In [ ]:
print("\n第三步 · 限幅：上一步 0.16，模型突然要 0.40，每步最多变 0.05")
print("  实际发出:", round(limit_step(0.16, 0.40, 0.05), 3))

In [ ]:
print("\n第三步（续）· 测量：看平均，更要看 p99")
latency_ms = [42, 45, 44, 41, 43, 46, 44, 120, 45, 43] * 10  # 100 个样本，每 10 个里有一个慢的（编的）
print(f"  平均 {sum(latency_ms) / len(latency_ms):.1f} ms，p50 {percentile(latency_ms, 50)} ms，"
      f"p99 {percentile(latency_ms, 99)} ms")

## 3. 代码里才看得到的

1. **`+ 1e-9`**：`(1.02 - 1.00) / 0.02` 在浮点里算出来是 0.99999…，不加一个极小的数会被 floor 成 0。时间戳换算成步数时到处都有这个坑。
2. **时间集成用了等权平均**：ACT 用指数权重 $w_i=e^{-m\,i}$（i 是块的新旧次序），这里简化成等权。
3. **平均值会骗人**：100 个样本里 10 个是 120 ms，平均只有 51 ms，p99 却是 120 ms。机器人卡顿看的是 p99 和抖动。
4. **所有时间戳、动作块、延迟样本都是编的**。超时停车那条规则这里没有演示，留给练习 20-4。

## 4. 自己改一改

- 把同步的容差 `slop` 改成 0.01，看 0.100 那张图能不能配上；
- 把限幅的 `max_delta` 改成 0.1，看实际发出多少。

在下面的空格子里写，或者直接改上面的格子。讲义末尾的练习先在纸上做，不在这里给答案。

In [ ]:
# 在这里写你的代码

## 口诀

- **输入**：图像 Best effort，时间戳是采集时刻，近似时间同步；坐标对齐 base_link。
- **快慢分离**：推理慢、执行快，分组分线程，中间一个短锁缓冲区。
- **衔接**：块与块之间要平滑（时间集成 / RTC），不要硬切。
- **输出**：短轨迹给 JTC，末端速度给 Servo，底盘速度给 diff_drive。
- **安全**：限幅、过期就停、输入断了就停、关键节点退出全停、急停在硬件。
- **托管**：configure 加载模型，activate 才动，deactivate 保持不动。